# Sleep by hour → Prism

Folder of SPINDLE CSVs + `start_times.csv` → average-day sleep table (2-hour bins, rotated to
start at 6 AM, averaged across days) → Prism workbook and group plots.

For a step-by-step explanation of every calculation, see `sleep_walkthrough.ipynb`.
Run top to bottom; only the **Settings** cell needs editing.

## Settings

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from openpyxl import Workbook

DATA_DIR = Path("/Users/leahmcintyre/Data/nri/Stxbp1_characterization")
SPINDLE_DIR = DATA_DIR / "spindle_outputs" / "LFC_RSC"      # one SPINDLE CSV per animal
START_TIMES_CSV = DATA_DIR / "start_times.csv"             # columns: animal_id, start_datetime
OUT_PATH = DATA_DIR / "sleep_export" / "prism_sleep_by_hour.xlsx"

EPOCH_SEC = 4        # SPINDLE epoch length
BIN_HOURS = 2
BEGIN_HR = 6         # clock hour that bin 0 (and ZT 0) starts at; 6 = 6 AM lights-on

# animal ID = filename text before the first "_"
groups = {
    "WT": [
        "Mice-C266", "Mice-C317", "Mice-C567", "Mice-C576", "Mice-C577",
        "Mice-1248", "Mice-1253", "Mice-1282", "Mice-1284", "Mice-1286", "Mice-1254",
    ],
    "MUT": [
        "Mice-C259", "Mice-C264", "Mice-C268", "Mice-C269", "Mice-C309", "Mice-C315",
        "Mice-C348", "Mice-C561", "Mice-C563", "Mice-C568", "Mice-C569",
        "Mice-1251", "Mice-1279",
    ],
}

## Compute

Per animal: read states → one state per second → rotate to the first `BEGIN_HR` → 2-hour
bins of fraction of time, bout count and mean bout length (seconds) per state.

In [ ]:
STATES = ["w", "n", "r"]
METRICS = ["frac_time", "n_bouts", "mean_bout_sec"]
STATE_NAMES = {"w": "Wake", "n": "NREM", "r": "REM"}


def load_states(path):
    cols = pd.read_csv(path, nrows=0).columns
    col = "rawstate" if "rawstate" in cols else "state"
    states = pd.read_csv(path, usecols=[col])[col].to_numpy()
    assert set(states) <= set(STATES), f"unexpected states in {path.name}"
    return states


def bouts(mask):
    """(number of runs of True, mean run length in samples)."""
    change = np.diff(mask.astype(int))
    starts = np.where(change == 1)[0] + 1
    ends = np.where(change == -1)[0] + 1
    if mask[0]:
        starts = np.r_[0, starts]
    if mask[-1]:
        ends = np.r_[ends, len(mask)]
    return (len(starts), (ends - starts).mean()) if len(starts) else (0, np.nan)


def seconds_until_next(start, hour):
    target = start.replace(hour=hour, minute=0, second=0, microsecond=0)
    if start > target:
        target += timedelta(days=1)
    return int((target - start).total_seconds())


def sleep_bins_for_animal(path, start):
    animal = path.name.split("_")[0]
    secs = np.repeat(load_states(path), EPOCH_SEC)
    offset = seconds_until_next(start, BEGIN_HR)
    if offset >= len(secs):
        print(f"skipping {animal}: recording ends before {BEGIN_HR}:00")
        return None
    rotated = np.concatenate([secs[offset:], secs[:offset]])
    bin_len = BIN_HOURS * 3600
    rows = []
    for i, s in enumerate(range(0, len(rotated), bin_len)):
        chunk = rotated[s:s + bin_len]
        for state in STATES:
            m = chunk == state
            n, mean_len = bouts(m)
            rows.append({"mouse_id": animal, "bin_start_hr": BEGIN_HR + i * BIN_HOURS,
                         "state": state, "frac_time": m.sum() / len(chunk),
                         "n_bouts": n, "mean_bout_sec": mean_len})
    return pd.DataFrame(rows)


starts = pd.read_csv(START_TIMES_CSV, parse_dates=["start_datetime"]).set_index("animal_id")["start_datetime"]
paths = sorted(SPINDLE_DIR.glob("*.csv"))
ids = [p.name.split("_")[0] for p in paths]
assert len(ids) == len(set(ids)), "two CSVs for one animal"
assert not set(ids) - set(starts.index), f"no start time: {sorted(set(ids) - set(starts.index))}"

per_animal = [sleep_bins_for_animal(p, starts[i]) for p, i in zip(paths, ids)]
bins = pd.concat([d for d in per_animal if d is not None], ignore_index=True)

# fold onto a 24 h day and average across days; ZT = hours since lights-on
bins["hour_of_day"] = (bins["bin_start_hr"] % 24).astype(int)
g = bins.groupby(["mouse_id", "state", "hour_of_day"], as_index=False)
avg = g[METRICS].mean()
avg["n_days"] = g.size()["size"]
avg["zt"] = (avg["hour_of_day"] - BEGIN_HR) % 24

animal_to_group = {a: grp for grp, animals in groups.items() for a in animals}
assert len(animal_to_group) == sum(map(len, groups.values())), "an animal is in two groups"
avg["group"] = avg["mouse_id"].map(animal_to_group)
unassigned = sorted(avg.loc[avg["group"].isna(), "mouse_id"].unique())
assert not unassigned, f"not in any group: {unassigned}"

print(avg["mouse_id"].nunique(), "animals")
avg.head()

## Write the Prism workbook

One sheet per state × metric (`w_frac_time`, `w_bouts`, `w_mean_bout_sec`, …): rows = ZT bins,
columns = animals blocked by group. Smaller groups are padded with blank columns.
`frac_time` is a fraction (0–1).

In [ ]:
ZT_LABEL = f"ZT hours (0 = {BEGIN_HR % 12 or 12} {'AM' if BEGIN_HR % 24 < 12 else 'PM'})"
SHEET_METRIC = {"n_bouts": "bouts"}


def write_prism_workbook(df, out_path):
    width = max(map(len, groups.values()))
    wb = Workbook()
    wb.remove(wb.active)
    for metric in METRICS:
        for state in STATES:
            wide = df[df["state"] == state].pivot(index="zt", columns="mouse_id", values=metric).sort_index()
            ws = wb.create_sheet(f"{state}_{SHEET_METRIC.get(metric, metric)}")
            ws.cell(row=2, column=1, value=ZT_LABEL)
            layout = []
            for grp, animals in groups.items():
                block = sorted(a for a in animals if a in wide.columns)
                block += [None] * (width - len(block)) if block else []
                layout += [(grp, a) for a in block]
            for c, (grp, animal) in enumerate(layout, start=2):
                ws.cell(row=1, column=c, value=grp)
                ws.cell(row=2, column=c, value=animal)
            for r, zt in enumerate(wide.index, start=3):
                ws.cell(row=r, column=1, value=f"{zt}-{zt + BIN_HOURS}")
                for c, (_, animal) in enumerate(layout, start=2):
                    val = wide.at[zt, animal] if animal in wide.columns else None
                    ws.cell(row=r, column=c, value=None if pd.isna(val) else float(val))
            ws.freeze_panes = "B3"
    wb.save(out_path)


OUT_PATH.parent.mkdir(exist_ok=True)
write_prism_workbook(avg, OUT_PATH)
print("Wrote", OUT_PATH)

## Plot

Group mean ± SEM per ZT bin; grey band = lights-off (ZT 12–24).

In [ ]:
ROWS = [("n_bouts", "Bout number", "bouts per bin", 1),
        ("mean_bout_sec", "Bout duration", "seconds", 1),
        ("frac_time", "Time in state", "% of bin", 100)]

fig, axes = plt.subplots(3, 3, figsize=(12, 9), sharex=True)
for r, (metric, title, ylabel, scale) in enumerate(ROWS):
    for c, state in enumerate(STATES):
        ax = axes[r, c]
        ax.axvspan(12, 24, color="0.88", zorder=0)
        for grp in groups:
            d = avg[(avg["state"] == state) & (avg["group"] == grp)]
            a = d.groupby("zt")[metric].agg(["mean", "sem"])
            ax.errorbar(a.index, a["mean"] * scale, yerr=a["sem"] * scale,
                        marker="o", capsize=3, label=grp)
        ax.set_title(f"{STATE_NAMES[state]}: {title}")
        ax.set_xlim(0, 24)
        ax.set_xticks(range(0, 25, 4))
        if c == 0:
            ax.set_ylabel(ylabel)
        if r == 2:
            ax.set_xlabel(ZT_LABEL)
axes[0, 0].legend()
fig.tight_layout();